# Standardizing and combining
* We are going to put all trading days into a standard format and now combine them into one big parquet file that contains info on every stock over the entire period of time being analysed.

In [1]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.append('/Users/manishkumar/quant')

import os
import pandas as pd
from src.standardize import standardize, validate

RAW = '/Users/manishkumar/quant/data/raw'
PROCESSED = '/Users/manishkumar/quant/data/processed'

In [2]:
good_days = []
bad_days = []
for fname in sorted(os.listdir(RAW)):
    if fname.endswith('.csv'):
        path = os.path.join(RAW, fname)
        try:
            df = standardize(path)
            validate(df)
            good_days.append(df)
        except Exception as e:
            bad_days.append((fname, type(e).__name__, str(e)))
            
print(len(good_days), len(bad_days))

4666 0


In [3]:
all_trading_records = pd.concat(good_days, ignore_index = True)
all_trading_records = all_trading_records.sort_values(['date', 'symbol', 'series']).reset_index(drop = True)
del good_days

In [4]:
all_trading_records.shape

(9005038, 14)

* Here we do a handover check between new format and old format of bhavcopy. In 2024 when the format changed and the new format had previous close price included, so we here are checking if these values match up on the format change day as a sanity check.

In [5]:
last_old = all_trading_records.loc[all_trading_records['fmt'] == 2, 'date'].max()
first_new = all_trading_records.loc[all_trading_records['fmt'] == 3, 'date'].min()

a = all_trading_records.loc[all_trading_records['date'] == last_old, ['symbol', 'series', 'close']]
b = all_trading_records.loc[all_trading_records['date'] == first_new, ['symbol', 'series', 'prev_close']]
m = a.merge(b, on=['symbol', 'series'])

mismatch = (m['close'] - m['prev_close']).abs() > 0.01
print(last_old.date(), first_new.date(), len(m), mismatch.sum())

2024-07-05 2024-07-08 2684 0


In [6]:
all_trading_records.to_parquet(PROCESSED + '/bhavcopy_all.parquet', index=False)

In [7]:
cr = pd.read_parquet(PROCESSED + '/bhavcopy_all.parquet')
print(cr.shape)
print(cr.dtypes)

(9005038, 14)
date          datetime64[ns]
symbol        string[python]
series        string[python]
isin          string[python]
open                 float64
high                 float64
low                  float64
close                float64
last                 float64
prev_close           float64
volume                 int64
value                float64
trades               float64
fmt                    int64
dtype: object
